In [26]:
%load_ext autoreload
%autoreload 2
import anndata as ad
import os
import pandas as pd
import scanpy as sc
from pipeline.utils import plot
from pipeline.utils.env import find_env_dir

pre_h5ad_dir = find_env_dir("PRE_H5AD")
clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")

SERIES = "ibd"

adata = sc.read_h5ad(os.path.join(clustered_h5ad_dir, SERIES + ".h5ad"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
kinchen_adata = sc.read_h5ad(os.path.join(pre_h5ad_dir, "kinchen_raw.h5ad"))
smillie_adata = sc.read_h5ad(os.path.join(pre_h5ad_dir, "smillie_raw.h5ad"))
kong_adata = sc.read_h5ad(os.path.join(pre_h5ad_dir, "kong_raw.h5ad"))
adata = ad.concat([kinchen_adata, smillie_adata, kong_adata], join="inner")


In [ ]:
adata.write_h5ad(os.path.join(pre_h5ad_dir, SERIES + "_raw.h5ad"))

In [19]:
cell_marker_genes = {
    "EntericNeuron": ["ELAVL4", "SNAP25", "CHGB", ],
    "EntericGlia": ["NGFR", "CDH19", "ASPA", "GPM6B", "SOX10", "PLP1", ],
    "Enteroendocrine": ["NKX2-2", "INSM1", "RFX6", "RIMBP2", "VWA5B2", "PAX6", "NEUROD1", "FEV", "ARX"],
    "Epithelial": ["HNF4A", "NXPE1", "EPCAM", "KRT8", "KRT18"],
    "ICC": ["KIT", "ANO1", ], 
    "Tuft": ["IL17RB", "POU2F3", "HMX2", ],
    "Goblet": ["MUC2", "SPINK4", "SPDEF", "TBX10"],
    "Adipocyte": ["PLIN1", "LPL"],
    "Endothelial": ["VWF", "CDH5", "FLT1", "CLDN5", "CLEC14A", "ERG", ],
    "LymphaticEndothelial": ["PROX1", "RELN", "CD36", "LYVE1"], 
    "Macrophage": ["C1QA", "C1QB", "C1QC", "FCGR2A", "CD163", "ITGAM", "LST1"],
    "Monocyte": ["CD300E", "FCN1", "CLEC12A", ],
    "MastCell": ["TPSAB1", "CPA3", "HDC", "GATA2", ], 
    "Fibroblast": ["LUM", "FBLN1", "FBLN2", "PDGFRA", "DCN", "KCNN3"],
    "Pericyte": ["PDGFRB", "NOTCH3", "RGS5", "STEAP4", ], 
    "Myocyte": ["MYH11", "SMTN", "LMOD1", "CNN1", "ACTA2", "TAGLN", ],
    "Tcell": ["IL7R", "IKZF1", "ITGA4", "PTPN22", ],
    "Mesothelial": ["MSLN", "GATA6", "MUC16", "CALB2", "PDPN", ],
}
plot.plot_dotplot(adata, "ibd", cell_marker_genes, group = "leiden")

In [ ]:
leiden_to_celltype = {
    "0": "Macrophage",
    "1": "?",
    "2": "Macrophage",
    "3": "?",
    "4": "?",
    "5": "?",
    "6": "?",
    "7": "EntericGlia",
    "8": "?",
    "9": "?",
    "10": "?",
    "11": "?",
    "12": "?",
    "13": "?",
    "14": "?",
    "15": "MastCell",
    "16": "?",
    "17": "?",
    "18": "?",
    "19": "?",
    "20": "Macrophage",
    "21": "Endothelial",
    "22": "?",
    "23": "?",
    "24": "?",
    "25": "?",
    "26": "?",
    "27": "?",
    "28": "?",
    "29": "?",
    "30": "?",
    "31": "?",
    "32": "?",
    "33": "?",
    "34": "Myocyte",
    "35": "?",
    "36": "Fibroblast",
    "37": "?",
    "38": "MastCell",
    "39": "Endothelial",
    "40": "LymphaticEndothelial",
    "41": "?",
    "42": "?",
    "43": "Pericyte",
    "44": "?",
    "45": "?",
    "46": "?",
    "47": "?",
    "48": "?",
    "49": "Tuft",
    "50": "?",
    "51": "Fibroblast",
    "52": "Fibroblast",
    "53": "Fibroblast",
    "54": "?",
    "55": "?",
    "56": "Macrophage",
    "57": "?",
    "58": "?",
    "59": "?",
    "60": "Fibroblast",
    "61": "?",
    "62": "?",
    "63": "?", 
    "64": "?",
    "65": "Monocyte",
    "66": "?",
    "67": "?",
    "68": "?",
    "69": "?",
    "70": "?",
    "71": "?",
    "72": "?",
    "73": "Doublet",
    "74": "?",
    "75": "?",
    "76": "?",
}

leiden_str = adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
adata.obs["cell"] = mapped

adata = adata[adata.obs["cell"] != "Doublet"].copy()

In [31]:
adata.write_h5ad(os.path.join(pre_h5ad_dir, "ibd.h5ad"))

In [ ]:
adata["celltype"] = adata.obs["cell"]

In [15]:
adata[adata.obs["leiden"] == "49"].obs["celltype"].value_counts()


celltype
Tuft cells                  1197
Tuft                         342
Stem cells OLFM4              27
Mast cells                    14
TA 1                          10
Enterocytes CA1 CA2 CA4-       8
T cells CD4 FOSB               6
Stem cells OLFM4 LGR5          3
T cells Naive CD4              3
Secretory TA                   3
Goblet cells SPINK4            2
Plasma                         2
Endothelial cells CD36         2
Enteroendocrine                2
T cells CD4 IL17A              2
CD69+ Mast                     1
Goblet cells MUC2 TFF1-        1
Epithelial Cycling cells       1
Enteroendocrine cells          1
CD4+ Memory                    1
Stem                           1
Paneth cells                   1
M cells                        1
Macrophages                    1
Name: count, dtype: int64

In [18]:
adata[adata.obs["cell"] == "MastCell"].obs["celltype"].value_counts()


celltype
Mast cells                    9110
CD69+ Mast                    5477
Plasma                         660
Plasma cells                   166
CD69- Mast                     138
CD4+ Activated Fos-hi           21
T cells CD4 FOSB                15
T cells OGT                     14
TA 1                            11
Tuft cells                      11
Macrophages                     10
B cells                         10
CD4+ Activated Fos-lo            7
Fibroblasts ADAMDEC1             7
Cycling TA                       6
Follicular                       6
Immune Cycling cells             6
Cycling Monocytes                5
CD8+ LP                          4
CD4+ Memory                      4
Myofibroblasts HHIP NPNT         4
Enterocytes CA1 CA2 CA4-         4
Stem cells OLFM4                 3
Cycling cells                    3
Goblet cells MUC2 TFF1           3
T cells CD8                      3
NK cells KLRF1 CD3G-             3
ILCs                             3
Macrophages

In [14]:
enteric_glia = adata[adata.obs["cell"] == "EntericGlia"]
enteric_glia.write_h5ad(os.path.join(pre_h5ad_dir,  SERIES + "_glia_raw.h5ad"))

/home/sjkim/protocols/.pixi/envs/default/lib/python3.13/site-packages/anndata/_core/anndata.py:1208: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
